# DecodeLabs AI Project 2: Student Performance Classification

This notebook uses **Logistic Regression** to predict whether a student's result is likely to be pass or fail from four study-related features. Work through the cells from top to bottom.

## 1. Load and Understand the Dataset

The CSV contains `study_hours`, `attendance`, `assignments`, and `previous_score` as input features. The `result` column is the label the model will learn to predict.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Support running with either the project root or notebooks/ as the working directory.
dataset_path = Path("../data/student_classification_dataset.csv")
if not dataset_path.exists():
    dataset_path = Path("data/student_classification_dataset.csv")

data = pd.read_csv(dataset_path)
print(f"Loaded {len(data)} student records from {dataset_path}")
data.head()

In [ ]:
print("Dataset dimensions (rows, columns):", data.shape)
print("\nColumn data types and non-null counts:")
data.info()
print("\nFeature summary:")
display(data.describe())
print("\nMissing values by column:")
display(data.isna().sum())
print("\nPass/fail counts:")
display(data["result"].value_counts())

## 2. Choose Features and Split the Data

`X` contains the four inputs and `y` contains the result to predict. We reserve 20% of the records for testing. `stratify=y` keeps the Pass/Fail proportions similar in both sets, which is useful for this small dataset.

In [ ]:
feature_columns = ["study_hours", "attendance", "assignments", "previous_score"]
X = data[feature_columns]
y = data["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training records: {len(X_train)}")
print(f"Testing records: {len(X_test)}")
print("Training label counts:\n", y_train.value_counts())
print("Testing label counts:\n", y_test.value_counts())

## 3. Train the Logistic Regression Model

The model finds a relationship between the student features and the known Pass/Fail labels in the training set. `max_iter=1000` gives the solver enough iterations to converge if needed.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Model training complete.")

## 4. Evaluate on the Test Set

The test records were held back during training. Accuracy is the fraction predicted correctly. The confusion matrix and classification report show how the model performs for each class; the matrix labels below are ordered as `Fail`, then `Pass`.

In [ ]:
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nConfusion matrix (rows = actual, columns = predicted; order: Fail, Pass):")
print(confusion_matrix(y_test, y_pred, labels=["Fail", "Pass"]))
print("\nClassification report:")
print(classification_report(y_test, y_pred, labels=["Fail", "Pass"], zero_division=0))

## 5. Predict for a New Student

Enter one example student's values using the same feature names and units as the training data. The model returns its predicted class.

In [ ]:
new_student = pd.DataFrame(
    [{"study_hours": 5, "attendance": 82, "assignments": 7, "previous_score": 68}],
    columns=feature_columns,
)
predicted_result = model.predict(new_student)[0]
print("Predicted result for the example student:", predicted_result)

## Conclusion

This workflow loads data, separates features and target, creates training and test sets, trains a classifier, predicts unseen outcomes, and evaluates those predictions. The dataset contains only 30 records, so the reported metrics are for learning and demonstration; they are not reliable evidence for real academic decisions.